In [ ]:
import customtkinter as ctk
import time
import folium
from tkinterweb import HtmlFrame
import os
import tkintermapview
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests

geolocator = Nominatim(user_agent="my_order_app")

def get_location(place_name):
    location = geolocator.geocode(place_name + ", Việt Nam")

    if location:
        return (location.latitude, location.longitude), location.address

    return None, None


def get_real_route(start_coord, end_coord):
    start_lat, start_lon = start_coord
    end_lat, end_lon = end_coord

    url = (
        f"https://router.project-osrm.org/route/v1/driving/"
        f"{start_lon},{start_lat};{end_lon},{end_lat}"
        f"?overview=full&geometries=geojson"
    )

    response = requests.get(url)
    data = response.json()

    if data["code"] != "Ok":
        return None, None, None

    route = data["routes"][0]

    distance_km = route["distance"] / 1000
    duration_min = route["duration"] / 60

    route_points = []
    for lon, lat in route["geometry"]["coordinates"]:
        route_points.append((lat, lon))

    return route_points, distance_km, duration_min
ctk.set_appearance_mode("light")
ctk.set_default_color_theme("blue")

app = ctk.CTk()
app.geometry("430x760")
app.title("Dmove")

# ================= PHONE FRAME =================
phone = ctk.CTkFrame(
    app,
    width=390,
    height=720,
    corner_radius=35,
    fg_color="white",
    border_width=6,
    border_color="#111827"
)
phone.place(x=25,y=0)
phone.pack_propagate(False)

# ================= STATUS BAR =================
status_frame = ctk.CTkFrame(phone, fg_color="transparent")
status_frame.pack(fill="x", padx=25, pady=(15, 5))

time_label = ctk.CTkLabel(
    status_frame,
    text="",
    font=("Arial", 14, "bold"),
    text_color="#111827"
)
time_label.pack(side="left")

ctk.CTkLabel(
    status_frame,
    text="WiFi 🔋",
    font=("Arial", 13, "bold"),
    text_color="#111827"
).pack(side="right")

def update_time():
    current_time = time.strftime("%H:%M")
    time_label.configure(text=current_time)
    app.after(1000, update_time)

update_time()
# ================= HEADER =================
header = ctk.CTkFrame(phone, fg_color="transparent")
header.pack(fill="x", padx=25, pady=(10, 5))

logo = ctk.CTkLabel(
    header,
    text="🛵",
    width=42,
    height=42,
    fg_color="#4DB7FF",
    corner_radius=12,
    font=("Arial", 22)
)
logo.pack(side="left")

ctk.CTkLabel(
    header,
    text="Dmove",
    font=("Arial", 26, "bold"),
    text_color="#1F2937"
).pack(side="left", padx=10)

ctk.CTkLabel(
    header,
    text="● Trực tuyến",
    font=("Arial", 13, "bold"),
    text_color="#16A34A",
    fg_color="#DCFCE7",
    corner_radius=20,
    width=100,
    height=32
).pack(side="right")

# ================= TITLE =================
ctk.CTkLabel(
    phone,
    text="Xin chào 👋",
    font=("Arial", 15),
    text_color="#8A8F98"
).pack(anchor="w", padx=30, pady=(10, 0))

ctk.CTkLabel(
    phone,
    text="Bạn muốn đi đâu hôm nay?",
    font=("Arial", 22, "bold"),
    text_color="#111827"
).pack(anchor="w", padx=30, pady=(0, 15))

# ================= SELECT VEHICLE TITLE =================
select_frame = ctk.CTkFrame(
    phone,
    fg_color="transparent"
)
select_frame.pack(fill="x", padx=30, pady=(28, 12))

ctk.CTkLabel(
    select_frame,
    text="Chọn phương tiện",
    font=("Arial", 24, "bold"),
    text_color="#082560"
).pack(anchor="w")

ctk.CTkLabel(
    select_frame,
    text="Chọn loại xe phù hợp với chuyến đi của bạn",
    font=("Arial", 14),
    text_color="#6B7280"
).pack(anchor="w", pady=(3, 0))

# ================= VEHICLE CARD =================
def open_motorbike_page():
    app.withdraw()

    motor_win = ctk.CTkToplevel()
    motor_win.geometry("430x820")
    motor_win.title("GoRide - Xe máy")
    motor_win.configure(fg_color="white")

    def back():
        motor_win.destroy()
        app.deiconify()

    ctk.CTkButton(
        motor_win,
        text="←",
        width=35,
        height=35,
        font=("Arial", 20, "bold"),
        fg_color="#F3F4F6",
        hover_color="#E5E7EB",
        text_color="#111827",
        corner_radius=18,
        command=back
    ).place(x=20, y=20)

    ctk.CTkLabel(
        motor_win,
        text="Di chuyển",
        font=("Arial", 30, "bold"),
        text_color="#111827"
    ).place(x=70, y=22)

    # ===== SEARCH BOX NẰM TRONG TAB XE MÁY =====
    # ===== LOCATION BOX =====
    location_frame = ctk.CTkFrame(
        motor_win,
        fg_color="white",
        corner_radius=24,
        width=370,
        height=130,
        border_width=1,
        border_color="#E5E7EB"
    )
    location_frame.place(x=30, y=105)
    location_frame.pack_propagate(False)

    # Ô điểm đón
    pickup_frame = ctk.CTkFrame(
        location_frame,
        fg_color="#F8FAFC",
        corner_radius=16,
        height=48
    )
    pickup_frame.pack(fill="x", padx=14, pady=(14, 6))
    pickup_frame.pack_propagate(False)

    ctk.CTkLabel(
        pickup_frame,
        text="●",
        text_color="#10B981",
        font=("Arial", 18, "bold")
    ).pack(side="left", padx=(14, 10))

    pickup_entry = ctk.CTkEntry(
        pickup_frame,
        placeholder_text="Bạn đang ở đâu?",
        border_width=0,
        fg_color="#F8FAFC",
        text_color="#111827",
        placeholder_text_color="#9CA3AF",
        font=("Arial", 15, "bold")
    )
    pickup_entry.pack(side="left", fill="x", expand=True)

    # Đường ngăn cách
    ctk.CTkFrame(
        location_frame,
        height=1,
        fg_color="#E5E7EB"
    ).pack(fill="x", padx=35, pady=2)

    # Ô điểm đến
    dropoff_frame = ctk.CTkFrame(
        location_frame,
        fg_color="#F8FAFC",
        corner_radius=16,
        height=48
    )
    dropoff_frame.pack(fill="x", padx=14, pady=(6, 14))
    dropoff_frame.pack_propagate(False)

    ctk.CTkLabel(
        dropoff_frame,
        text="●",
        text_color="#2563EB",
        font=("Arial", 18, "bold")
    ).pack(side="left", padx=(14, 10))

    dropoff_entry = ctk.CTkEntry(
        dropoff_frame,
        placeholder_text="Bạn muốn đến đâu?",
        border_width=0,
        fg_color="#F8FAFC",
        text_color="#111827",
        placeholder_text_color="#9CA3AF",
        font=("Arial", 15, "bold")
    )
    dropoff_entry.pack(side="left", fill="x", expand=True)
        # ===== MAP THẬT - TP.HCM =====
    map_frame = ctk.CTkFrame(
        motor_win,
        width=370,
        height=260,
        corner_radius=24,
        fg_color="white",
        border_width=1,
        border_color="#E5E7EB"
    )
    map_frame.place(x=30, y=255)
    map_frame.pack_propagate(False)

    map_widget = tkintermapview.TkinterMapView(
        map_frame,
        width=350,
        height=240,
        corner_radius=18
    )
    map_widget.pack(padx=10, pady=10)

    # Vị trí TP.HCM
    map_widget.set_tile_server("https://a.tile.openstreetmap.org/{z}/{x}/{y}.png")
    map_widget.set_position(10.7769, 106.7009)
    map_widget.set_zoom(13)
    # ===== FUNCTION HIỂN THỊ ĐƯỜNG ĐI =====
    def show_route_on_map():
        start_name = pickup_entry.get()
        end_name = dropoff_entry.get()

        start_coord, start_address = get_location(start_name)
        end_coord, end_address = get_location(end_name)

        if not start_coord or not end_coord:
            print("Không tìm thấy địa điểm")
            return

        map_widget.delete_all_marker()
        map_widget.delete_all_path()

        map_widget.set_marker(start_coord[0], start_coord[1], text="Điểm đầu")
        map_widget.set_marker(end_coord[0], end_coord[1], text="Điểm cuối")

        route_points, distance, duration = get_real_route(start_coord, end_coord)

        if not route_points:
            print("Không tìm được đường đi thật")
            return

        map_widget.set_path(route_points, color="#2563EB", width=6)

        center_lat = (start_coord[0] + end_coord[0]) / 2
        center_lon = (start_coord[1] + end_coord[1]) / 2

        map_widget.set_position(center_lat, center_lon)
        map_widget.set_zoom(13)

        # ===== TÍNH TIỀN XE MÁY =====
        # Công thức: 10.000 + 4.000 × số km
        fare = 10000 + 4000 * distance
        fare_formatted = f"{int(fare):,}".replace(",", ".")

        # Hiện thông tin chuyến đi
        info_label.configure(
            text=(
                f"📍 {round(distance, 2)} km   ⏱ {round(duration)} phút   "
                f"💰 {fare_formatted} đ"
            )
        )
        info_frame.configure(fg_color="#EFF6FF", border_color="#BFDBFE")

        # Cập nhật nút đặt xe với giá tiền
        confirm_btn.configure(
            text=f"✅ Xác nhận đặt xe – {fare_formatted} đ"
        )

    # ===== THÔNG TIN CHUYẾN ĐI =====
    info_frame = ctk.CTkFrame(
        motor_win,
        width=370,
        height=48,
        corner_radius=14,
        fg_color="#F9FAFB",
        border_width=1,
        border_color="#E5E7EB"
    )
    info_frame.place(x=30, y=530)
    info_frame.pack_propagate(False)

    info_label = ctk.CTkLabel(
        info_frame,
        text="Nhập điểm đón & điểm đến rồi bấm Đặt Xe",
        font=("Arial", 13),
        text_color="#6B7280"
    )
    info_label.pack(expand=True)

    # ===== BUTTON TÌM ĐƯỜNG =====
    ctk.CTkButton(
        motor_win,
        text="🔍 Đặt Xe",
        width=370,
        height=50,
        corner_radius=18,
        fg_color="#2563EB",
        hover_color="#1D4ED8",
        text_color="white",
        font=("Arial", 16, "bold"),
        command=show_route_on_map
    ).place(x=30, y=592)

    # ===== NÚT XÁC NHẬN =====
    confirm_btn = ctk.CTkButton(
        motor_win,
        text="✅ Xác nhận đặt xe",
        width=370,
        height=50,
        corner_radius=18,
        fg_color="#16A34A",
        hover_color="#15803D",
        text_color="white",
        font=("Arial", 15, "bold"),
        command=lambda: ctk.CTkLabel(
            motor_win,
            text="🎉 Đã đặt xe thành công! Tài xế đang đến...",
            font=("Arial", 14, "bold"),
            text_color="#16A34A",
            fg_color="#DCFCE7",
            corner_radius=12,
            width=370,
            height=40
        ).place(x=30, y=755)
    )
    confirm_btn.place(x=30, y=655)
vehicle1= ctk.CTkButton(
phone,
text="🛵\nXe máy",
width=340,
height=120,
corner_radius=30,
fg_color="#72DCD6",        # nền xanh nhạt giống hình
hover_color="#1CC0B0",
text_color="#111827",
font=("Arial", 24, "bold"),
command=open_motorbike_page
)

vehicle1.place(x=25, y=300)

def open_car_page():
    app.withdraw()

    car_win = ctk.CTkToplevel()
    car_win.geometry("430x820")
    car_win.title("Dmove - Xe hơi")
    car_win.configure(fg_color="white")

    def back():
        car_win.destroy()
        app.deiconify()

    ctk.CTkButton(
        car_win,
        text="←",
        width=35,
        height=35,
        font=("Arial", 20, "bold"),
        fg_color="#F3F4F6",
        hover_color="#E5E7EB",
        text_color="#111827",
        corner_radius=18,
        command=back
    ).place(x=20, y=20)

    ctk.CTkLabel(
        car_win,
        text="Di chuyển",
        font=("Arial", 30, "bold"),
        text_color="#111827"
    ).place(x=70, y=22)

    # ===== LOCATION BOX =====
    location_frame = ctk.CTkFrame(
        car_win,
        fg_color="white",
        corner_radius=24,
        width=370,
        height=130,
        border_width=1,
        border_color="#E5E7EB"
    )
    location_frame.place(x=30, y=105)
    location_frame.pack_propagate(False)

    pickup_frame = ctk.CTkFrame(
        location_frame,
        fg_color="#F8FAFC",
        corner_radius=16,
        height=48
    )
    pickup_frame.pack(fill="x", padx=14, pady=(14, 6))
    pickup_frame.pack_propagate(False)

    ctk.CTkLabel(
        pickup_frame,
        text="●",
        text_color="#10B981",
        font=("Arial", 18, "bold")
    ).pack(side="left", padx=(14, 10))

    pickup_entry = ctk.CTkEntry(
        pickup_frame,
        placeholder_text="Bạn đang ở đâu?",
        border_width=0,
        fg_color="#F8FAFC",
        text_color="#111827",
        placeholder_text_color="#9CA3AF",
        font=("Arial", 15, "bold")
    )
    pickup_entry.pack(side="left", fill="x", expand=True)

    ctk.CTkFrame(
        location_frame,
        height=1,
        fg_color="#E5E7EB"
    ).pack(fill="x", padx=35, pady=2)

    dropoff_frame = ctk.CTkFrame(
        location_frame,
        fg_color="#F8FAFC",
        corner_radius=16,
        height=48
    )
    dropoff_frame.pack(fill="x", padx=14, pady=(6, 14))
    dropoff_frame.pack_propagate(False)

    ctk.CTkLabel(
        dropoff_frame,
        text="●",
        text_color="#2563EB",
        font=("Arial", 18, "bold")
    ).pack(side="left", padx=(14, 10))

    dropoff_entry = ctk.CTkEntry(
        dropoff_frame,
        placeholder_text="Bạn muốn đến đâu?",
        border_width=0,
        fg_color="#F8FAFC",
        text_color="#111827",
        placeholder_text_color="#9CA3AF",
        font=("Arial", 15, "bold")
    )
    dropoff_entry.pack(side="left", fill="x", expand=True)

    # ===== BẢN ĐỒ =====
    map_frame = ctk.CTkFrame(
        car_win,
        width=370,
        height=260,
        corner_radius=24,
        fg_color="white",
        border_width=1,
        border_color="#E5E7EB"
    )
    map_frame.place(x=30, y=255)
    map_frame.pack_propagate(False)

    map_widget = tkintermapview.TkinterMapView(
        map_frame,
        width=350,
        height=240,
        corner_radius=18
    )
    map_widget.pack(padx=10, pady=10)
    map_widget.set_tile_server("https://a.tile.openstreetmap.org/{z}/{x}/{y}.png")
    map_widget.set_position(10.7769, 106.7009)
    map_widget.set_zoom(13)

    # ===== HIỂN THỊ ĐƯỜNG ĐI + TÍNH TIỀN XE HƠI =====
    def show_route_on_map():
        start_name = pickup_entry.get()
        end_name = dropoff_entry.get()

        start_coord, _ = get_location(start_name)
        end_coord, _ = get_location(end_name)

        if not start_coord or not end_coord:
            info_label.configure(text="⚠️ Không tìm thấy địa điểm, thử lại nhé!")
            info_frame.configure(fg_color="#FEF2F2", border_color="#FECACA")
            return

        map_widget.delete_all_marker()
        map_widget.delete_all_path()
        map_widget.set_marker(start_coord[0], start_coord[1], text="Điểm đầu")
        map_widget.set_marker(end_coord[0], end_coord[1], text="Điểm cuối")

        route_points, distance, duration = get_real_route(start_coord, end_coord)

        if not route_points:
            info_label.configure(text="⚠️ Không tìm được đường đi!")
            return

        map_widget.set_path(route_points, color="#DC2626", width=6)

        center_lat = (start_coord[0] + end_coord[0]) / 2
        center_lon = (start_coord[1] + end_coord[1]) / 2
        map_widget.set_position(center_lat, center_lon)
        map_widget.set_zoom(13)

        # ===== TÍNH TIỀN XE HƠI =====
        # Công thức: 15.000 + 4.000 × số km
        fare = 15000 + 4000 * distance
        fare_formatted = f"{int(fare):,}".replace(",", ".")

        info_label.configure(
            text=(
                f"📍 {round(distance, 2)} km   ⏱ {round(duration)} phút   "
                f"💰 {fare_formatted} đ"
            )
        )
        info_frame.configure(fg_color="#FFF1F2", border_color="#FECDD3")

        confirm_btn.configure(
            text=f"✅ Xác nhận đặt xe – {fare_formatted} đ"
        )

    # ===== THÔNG TIN CHUYẾN ĐI =====
    info_frame = ctk.CTkFrame(
        car_win,
        width=370,
        height=48,
        corner_radius=14,
        fg_color="#F9FAFB",
        border_width=1,
        border_color="#E5E7EB"
    )
    info_frame.place(x=30, y=530)
    info_frame.pack_propagate(False)

    info_label = ctk.CTkLabel(
        info_frame,
        text="Nhập điểm đón & điểm đến rồi bấm Đặt Xe",
        font=("Arial", 13),
        text_color="#6B7280"
    )
    info_label.pack(expand=True)

    # ===== BUTTON ĐẶT XE =====
    ctk.CTkButton(
        car_win,
        text="🔍 Đặt Xe",
        width=370,
        height=50,
        corner_radius=18,
        fg_color="#DC2626",
        hover_color="#B91C1C",
        text_color="white",
        font=("Arial", 16, "bold"),
        command=show_route_on_map
    ).place(x=30, y=592)

    # ===== NÚT XÁC NHẬN =====
    confirm_btn = ctk.CTkButton(
        car_win,
        text="✅ Xác nhận đặt xe",
        width=370,
        height=50,
        corner_radius=18,
        fg_color="#16A34A",
        hover_color="#15803D",
        text_color="white",
        font=("Arial", 15, "bold"),
        command=lambda: ctk.CTkLabel(
            car_win,
            text="🎉 Đã đặt xe thành công! Tài xế đang đến...",
            font=("Arial", 14, "bold"),
            text_color="#16A34A",
            fg_color="#DCFCE7",
            corner_radius=12,
            width=370,
            height=40
        ).place(x=30, y=755)
    )
    confirm_btn.place(x=30, y=655)

vehicle2= ctk.CTkButton(
    phone,
    text="🚗\nXe hơi",
    width=340,
    height=120,
    corner_radius=30,
    fg_color="#72DCD6",
    hover_color="#1CC0B0",
    text_color="#111827",
    font=("Arial", 24, "bold"),
    command=open_car_page
)
vehicle2.place(x=25, y=450)
app.mainloop()